In [2]:
import humanoid_bench
import gymnasium as gym
import numpy as np
import time
import cv2 
from time import sleep
from fast_td3 import ActorGNN
import humanoid_bench

In [ ]:
actor_detach = ActorGNN(
    n_obs=19,
    n_act=n_act,
    num_envs=args.num_envs,
    batch_size=args.batch_size,
    device=device,
    init_scale=args.init_scale,
    hidden_dim=64,
)

In [3]:
from fast_td3.egnn_clean import EGNN, get_edges_batch
import torch

batch_size = 4
n_nodes = 19
n_feat = 1
x_dim = 3

# Dummy variables h, x and fully connected edges
h = torch.ones(batch_size *  n_nodes, n_feat, device="cuda:0")
x = torch.ones(batch_size * n_nodes, x_dim, device="cuda:0")
edges, edge_attr = get_edges_batch(n_nodes, batch_size)
edges = [edge.to("cuda:0") for edge in edges]
# change edge_attr to have 3 dimensions
edge_attr = torch.ones(edges[0].shape[0] ,batch_size, device="cuda:0")

print("Node features shape:", h.shape)
print("Node positions shape:", x.shape)
print("Edges shape:", edges[0].shape)
print("Edge attributes shape:", edge_attr.shape)

# Initialize EGNN
egnn = EGNN(in_node_nf=n_feat, hidden_nf=32, out_node_nf=1, in_edge_nf=batch_size, batch_size=batch_size, device="cuda:0")
# Run EGNN
h = egnn(h, x, edges, edge_attr)

print("Output node features shape:", h.shape)
print("Output node positions shape:", x.shape)

Node features shape: torch.Size([76, 1])
Node positions shape: torch.Size([76, 3])
Edges shape: torch.Size([1368])
Edge attributes shape: torch.Size([1368, 4])
Output node features shape: torch.Size([4, 19])
Output node positions shape: torch.Size([76, 3])


In [4]:
import cv2
import torch
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

env = gym.make(
        "h1-stand-v0",
        render_mode="rgb_array",
)
env.reset()
data = env.unwrapped.named.data
# print(data.xpos)
# print(data.xipos)
# print(data.qvel)
# print(data.qpos)
# print(data.xanchor)

env = HumanoidBenchEnv(
        "h1-stand-v0",
        render_mode="rgb_array",
)

# observation, reward, terminated, truncated, info = env.step(torch.zeros(19, dtype=torch.float32))
# print(info)

# img = env.render()
# # save img
# cv2.imwrite("test.png", img)

actor_detach = ActorGNN(
    n_obs=19,
    n_act=env.num_actions,
    num_envs=16,
    hidden_dim=256,
    init_scale=0.01,
    device="cuda:0",
    batch_size=8912,
)

print("nunmber of parameters:", sum(p.numel() for p in actor_detach.parameters() if p.requires_grad))

nunmber of parameters: 1843201


In [4]:
env = gym.make(
        "h1hand-stand-v0",
        render_mode="rgb_array",
)
data = env.unwrapped.named.data
print(env.observation_space)
print(data.xanchor)

Box(-inf, inf, (151,), float64)
FieldIndexer(xanchor):
                          x         y         z         
 0            free_base [ 0         0         0       ]
 1         left_hip_yaw [ 0         0         0       ]
 2        left_hip_roll [ 0         0         0       ]
 3       left_hip_pitch [ 0         0         0       ]
 4            left_knee [ 0         0         0       ]
 5           left_ankle [ 0         0         0       ]
 6        right_hip_yaw [ 0         0         0       ]
 7       right_hip_roll [ 0         0         0       ]
 8      right_hip_pitch [ 0         0         0       ]
 9           right_knee [ 0         0         0       ]
10          right_ankle [ 0         0         0       ]
11                torso [ 0         0         0       ]
12  left_shoulder_pitch [ 0         0         0       ]
13   left_shoulder_roll [ 0         0         0       ]
14    left_shoulder_yaw [ 0         0         0       ]
15           left_elbow [ 0         0         0 

In [14]:
joint_names = [
    "left_hip_yaw",
    "left_hip_roll",
    "left_hip_pitch",
    "left_knee",
    "left_ankle",
    "right_hip_yaw",
    "right_hip_roll",
    "right_hip_pitch",
    "right_knee",
    "right_ankle",
    "torso",
    "left_shoulder_pitch",
    "left_shoulder_roll",
    "left_shoulder_yaw",
    "left_elbow",
    "right_shoulder_pitch",
    "right_shoulder_roll",
    "right_shoulder_yaw",
    "right_elbow",
]
joint_idx = {name: idx for idx, name in enumerate(joint_names)}
edge_list = [
    # Left leg
    (joint_idx["left_hip_yaw"], joint_idx["left_hip_roll"]),
    (joint_idx["left_hip_roll"], joint_idx["left_hip_pitch"]),
    (joint_idx["left_hip_pitch"], joint_idx["left_knee"]),
    (joint_idx["left_knee"], joint_idx["left_ankle"]),
    # Right leg
    (joint_idx["right_hip_yaw"], joint_idx["right_hip_roll"]),
    (joint_idx["right_hip_roll"], joint_idx["right_hip_pitch"]),
    (joint_idx["right_hip_pitch"], joint_idx["right_knee"]),
    (joint_idx["right_knee"], joint_idx["right_ankle"]),
    # Torso
    (joint_idx["torso"], joint_idx["left_hip_yaw"]),
    (joint_idx["torso"], joint_idx["right_hip_yaw"]),
    # Left arm
    (joint_idx["torso"], joint_idx["left_shoulder_pitch"]),
    (joint_idx["left_shoulder_pitch"], joint_idx["left_shoulder_roll"]),
    (joint_idx["left_shoulder_roll"], joint_idx["left_shoulder_yaw"]),
    (joint_idx["left_shoulder_yaw"], joint_idx["left_elbow"]),
    # Right arm
    (joint_idx["torso"], joint_idx["right_shoulder_pitch"]),
    (joint_idx["right_shoulder_pitch"], joint_idx["right_shoulder_roll"]),
    (joint_idx["right_shoulder_roll"], joint_idx["right_shoulder_yaw"]),
    (joint_idx["right_shoulder_yaw"], joint_idx["right_elbow"]),
]

print("Edge list:", edge_list)
print("Number of edges:", len(edge_list))

Edge list: [(0, 1), (1, 2), (2, 3), (3, 4), (5, 6), (6, 7), (7, 8), (8, 9), (10, 0), (10, 5), (10, 11), (11, 12), (12, 13), (13, 14), (10, 15), (15, 16), (16, 17), (17, 18)]
Number of edges: 18
